# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [116]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier, BaggingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
import numpy as np
from joblib import dump

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

In [8]:
sup_df = pd.read_csv("../data/dayofweek.csv")

In [9]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv')
df["dayofweek"] = sup_df["dayofweek"]
df

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1,dayofweek
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1681,9,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,3
1682,6,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,3
1683,7,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,3
1684,8,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,3


In [ ]:
X = df.drop('dayofweek', axis=1)
y = df["dayofweek"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)


In [15]:
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=21, stratify=y_train)

## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [17]:
model = SVC(C=10, class_weight=None, gamma='auto', kernel='rbf', random_state=21, probability=True)
model.fit(X_train, y_train)

y_pred = model.predict(X_valid)
accuracy = model.score(X_valid, y_valid)
precision = precision_score(y_valid, y_pred, average='weighted')
recall = recall_score(y_valid, y_pred, average='weighted')

print(f"accuracy is {accuracy:.5f}\nprecision is {precision:.5f}\nrecall is {recall:.5f}")

accuracy is 0.87778
precision is 0.88162
recall is 0.87778


In [19]:
model = DecisionTreeClassifier(class_weight='balanced', random_state=21, criterion='gini', max_depth=21)
model.fit(X_train, y_train)

y_pred = model.predict(X_valid)
accuracy = model.score(X_valid, y_valid)
precision = precision_score(y_valid, y_pred, average='weighted')
recall = recall_score(y_valid, y_pred, average='weighted')

print(f"accuracy is {accuracy:.5f}\nprecision is {precision:.5f}\nrecall is {recall:.5f}")

accuracy is 0.86667
precision is 0.87170
recall is 0.86667


In [21]:
model = RandomForestClassifier(class_weight='balanced', random_state=21, criterion='entropy', max_depth=24, n_estimators=100)
model.fit(X_train, y_train)

y_pred = model.predict(X_valid)
accuracy = model.score(X_valid, y_valid)
precision = precision_score(y_valid, y_pred, average='weighted')
recall = recall_score(y_valid, y_pred, average='weighted')

print(f"accuracy is {accuracy:.5f}\nprecision is {precision:.5f}\nrecall is {recall:.5f}")

accuracy is 0.89630
precision is 0.89698
recall is 0.89630


## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.
2. Play with the other parameteres.
3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

In [26]:
svm = SVC(C=10, class_weight=None, gamma='auto', kernel='rbf', random_state=21, probability=True)
tree = DecisionTreeClassifier(class_weight='balanced', random_state=21, criterion='gini', max_depth=21)
forest = RandomForestClassifier(class_weight='balanced', random_state=21, criterion='entropy', max_depth=24, n_estimators=100)

In [27]:
eclf = VotingClassifier(estimators=[('svm', svm), ('tree', tree), ('forest', forest)], voting='hard')
eclf.fit(X_train, y_train)

y_pred = eclf.predict(X_valid)
accuracy = eclf.score(X_valid, y_valid)
precision = precision_score(y_valid, y_pred, average='weighted')
recall = recall_score(y_valid, y_pred, average='weighted')

print(f"accuracy is {accuracy:.5f}\nprecision is {precision:.5f}\nrecall is {recall:.5f}")

accuracy is 0.90000
precision is 0.89993
recall is 0.90000


In [36]:
eclf = VotingClassifier(estimators=[('svm', svm), ('tree', tree), ('forest', forest)], voting='hard', weights=[1,1,4])
eclf.fit(X_train, y_train)

y_pred = eclf.predict(X_valid)
accuracy = eclf.score(X_valid, y_valid)
precision = precision_score(y_valid, y_pred, average='weighted')
recall = recall_score(y_valid, y_pred, average='weighted')

print(f"accuracy is {accuracy:.5f}\nprecision is {precision:.5f}\nrecall is {recall:.5f}")

accuracy is 0.89630
precision is 0.89698
recall is 0.89630


In [37]:
eclf = VotingClassifier(estimators=[('svm', svm), ('tree', tree), ('forest', forest)], voting='soft', weights=[4,1,4])
eclf.fit(X_train, y_train)

y_pred = eclf.predict(X_valid)
accuracy = eclf.score(X_valid, y_valid)
precision = precision_score(y_valid, y_pred, average='weighted')
recall = recall_score(y_valid, y_pred, average='weighted')

print(f"accuracy is {accuracy:.5f}\nprecision is {precision:.5f}\nrecall is {recall:.5f}")

accuracy is 0.91111
precision is 0.91288
recall is 0.91111


In [39]:
eclf = VotingClassifier(estimators=[('svm', svm), ('tree', tree), ('forest', forest)], voting='soft', weights=[4,1,4])
eclf.fit(X_train, y_train)

y_pred = eclf.predict(X_test)
accuracy = eclf.score(X_test, y_test)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print(f"accuracy is {accuracy:.5f}\nprecision is {precision:.5f}\nrecall is {recall:.5f}")

accuracy is 0.90533
precision is 0.90881
recall is 0.90533


## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.
2. Play with the other parameters.
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

In [80]:
model = SVC(C=10, class_weight=None, gamma='auto', kernel='rbf', random_state=21, probability=True)
bagging_model = BaggingClassifier(base_estimator=model, n_estimators=50, random_state=21, bootstrap=False, max_samples=0.95)
bagging_model.fit(X_train, y_train)

y_pred = bagging_model.predict(X_valid)
accuracy = bagging_model.score(X_valid, y_valid)
precision = precision_score(y_valid, y_pred, average='weighted')

print(accuracy, precision)


0.8888888888888888 0.8937816791667883


In [76]:
model = SVC(C=10, class_weight=None, gamma='auto', kernel='rbf', random_state=21, probability=True)
bagging_model = BaggingClassifier(base_estimator=model, n_estimators=63, random_state=21)
bagging_model.fit(X_train, y_train)

y_pred = bagging_model.predict(X_valid)
accuracy = bagging_model.score(X_valid, y_valid)
precision = precision_score(y_valid, y_pred, average='weighted')

print(accuracy, precision)

0.8851851851851852 0.8939608307494487


In [81]:
model = SVC(C=10, class_weight=None, gamma='auto', kernel='rbf', random_state=21, probability=True)
bagging_model = BaggingClassifier(base_estimator=model, n_estimators=50, random_state=21, bootstrap=False, max_samples=0.95)
bagging_model.fit(X_train, y_train)

y_pred = bagging_model.predict(X_test)
accuracy = bagging_model.score(X_test, y_test)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print(f"accuracy is {accuracy:.5f}\nprecision is {precision:.5f}\nrecall is {recall:.5f}")

accuracy is 0.88462
precision is 0.88831
recall is 0.88462


## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

In [ ]:
results = []
for n_splits in range(2, 8):
    for passthrough in [True, False]:
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=21)
        estimators = [('svm', SVC(C=10, class_weight=None, gamma='auto', kernel='rbf', random_state=21, probability=True)),
        ('tree', DecisionTreeClassifier(class_weight='balanced', random_state=21, criterion='gini', max_depth=21)),
        ('forest', RandomForestClassifier(class_weight='balanced', random_state=21, criterion='entropy', max_depth=24, n_estimators=100))]

        clf = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(solver='liblinear'), passthrough=passthrough, cv=skf)
        clf.fit(X_train, y_train)

        y_pred = clf.predict(X_valid)
        accuracy = clf.score(X_valid, y_valid)
        precision = precision_score(y_valid, y_pred, average='weighted')
        recall = recall_score(y_valid, y_pred, average='weighted')

        metrics = {"n_splits": n_splits,
                   'passthrough': passthrough,
                   'accuracy': accuracy,
                   'precision': precision,
                   'recall': recall}
        
        results.append(metrics)
        

results = pd.DataFrame(results)

In [99]:
results

,n_splits,passthrough,accuracy,precision,recall
0,2,True,0.903704,0.906190,0.903704
1,2,False,0.896296,0.896784,0.896296
2,3,True,0.903704,0.906322,0.903704
3,3,False,0.896296,0.897592,0.896296
4,4,True,0.911111,0.913269,0.911111
5,4,False,0.903704,0.905703,0.903704
6,5,True,0.900000,0.902167,0.900000
7,5,False,0.900000,0.900558,0.900000
8,6,True,0.903704,0.904500,0.903704
9,6,False,0.903704,0.904365,0.903704


In [100]:
skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=21)
clf = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(solver='liblinear'), passthrough=True, cv=skf)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
accuracy = clf.score(X_test, y_test)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print(f"accuracy is {accuracy:.5f}\nprecision is {precision:.5f}\nrecall is {recall:.5f}")

accuracy is 0.90533
precision is 0.90844
recall is 0.90533


## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.

In [109]:
skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=21)
estimators = [('svm', SVC(C=10, class_weight=None, gamma='auto', kernel='rbf', random_state=21, probability=True)),
        ('tree', DecisionTreeClassifier(class_weight='balanced', random_state=21, criterion='gini', max_depth=21)),
        ('forest', RandomForestClassifier(class_weight='balanced', random_state=21, criterion='entropy', max_depth=24, n_estimators=100))]
model = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(solver='liblinear'), passthrough=True, cv=skf)
model.fit(X_train, y_train)

y_pred_train = pd.Series(model.predict(X_train))
y_pred_valid = pd.Series(model.predict(X_valid))
y_pred_test = pd.Series(model.predict(X_test))


In [113]:
most_problematic_day = [0]*7

for i in range(y_test.shape[0]):
    if y_pred_test.iloc[i] != y_test.iloc[i]:
        most_problematic_day[y_test.iloc[i]] += 1

for i in range(y_train.shape[0]):
    if y_pred_train.iloc[i] != y_train.iloc[i]:
        most_problematic_day[y_train.iloc[i]] += 1

for i in range(y_valid.shape[0]):
    if y_pred_train.iloc[i] != y_valid.iloc[i]:
        most_problematic_day[y_valid.iloc[i]] += 1
        
max_error = max(most_problematic_day)
day_with_max_errors = [i for i in range(len(most_problematic_day)) if most_problematic_day[i] == max_error]

day_with_max_errors

[6]

In [115]:
most_problematic_user = {feature: 0 for feature in X_test.iloc[0, 2:32].index.tolist()}
most_problematic_labname = {feature: 0 for feature in X_test.iloc[0, 32:].index.tolist()}

incorrect_mask = y_pred_test.reset_index(drop=True) != y_test.reset_index(drop=True)
for i in incorrect_mask[incorrect_mask].index:
    row = X_test.iloc[i]
    cnt = 0
    for feature in X_test.columns[2:]:
        if cnt == 0 and row[feature] == 1:
            most_problematic_user[feature] += 1
            cnt += 1
        elif cnt != 0 and row[feature] == 1:
            most_problematic_labname[feature] += 1

incorrect_mask = y_pred_train.reset_index(drop=True) != y_train.reset_index(drop=True)
for i in incorrect_mask[incorrect_mask].index:
    row = X_train.iloc[i]
    cnt = 0
    for feature in X_train.columns[2:]:
        if cnt == 0 and row[feature] == 1:
            most_problematic_user[feature] += 1
            cnt += 1
        elif cnt != 0 and row[feature] == 1:
            most_problematic_labname[feature] += 1
        
incorrect_mask = y_pred_valid.reset_index(drop=True) != y_valid.reset_index(drop=True)
for i in incorrect_mask[incorrect_mask].index:
    row = X_valid.iloc[i]
    cnt = 0
    for feature in X_valid.columns[2:]:
        if cnt == 0 and row[feature] == 1:
            most_problematic_user[feature] += 1
            cnt += 1
        elif cnt != 0 and row[feature] == 1:
            most_problematic_labname[feature] += 1

max_error_user = max(most_problematic_user.values())
max_error_labname = max(most_problematic_labname.values())

user= next(key for key, value in most_problematic_user.items() if value == max_error_user)
labname= next(key for key, value in most_problematic_labname.items() if value == max_error_labname)

print(user, labname)

uid_user_2 labname_project1


In [117]:
dump(model, "../data/model_ex03.joblib", compress=9)

['../data/model_ex03.joblib']